# Lesson 4: Kernels and Covariance: the mathematical engine inside a GP

This lesson is extremely important because **the kernel is what determines how BoTorch believes two experimental conditions are related**.

For your eventual electrolyte SDL, this is the mechanism that answers:

> “If I have already measured electrolyte formulation A, how much should that measurement tell me about formulation B?”

The answer comes from the **covariance/kernel function**.

The current BoTorch documentation confirms that `SingleTaskGP` accepts a `covar_module` and, if you do not provide one, uses an `RBFKernel` in the current stable API.

## The big picture

Recall our GP:

$$
f(x)\sim\mathcal{GP}(m(x),k(x,x'))
$$

There are two major functions:

$$
\boxed{m(x)}
$$

the mean function, and

$$
\boxed{k(x,x')}
$$

the covariance/kernel function.

The kernel is much more important for understanding BO than it may initially appear.

Suppose your electrolyte formulation is represented by

$$
x=
[
x_{\mathrm{EC}},
x_{\mathrm{EMC}},
x_{\mathrm{LiPF6}},
x_{\mathrm{additive}}
]
$$

and you have two formulations:

$$
x_1=
[0.30,0.60,0.10,0.02]
$$

and

$$
x_2=
[0.32,0.58,0.10,0.02].
$$

The GP asks:

> How similar are $x_1$ and $x_2$?

The kernel returns a number:

$$
k(x_1,x_2).
$$

Large covariance:

$$
k(x_1,x_2)\rightarrow\text{large}
$$

means:

> “These points are strongly related.”

Small covariance:

$$
k(x_1,x_2)\rightarrow 0
$$

means:

> “Knowing the result at one tells me relatively little about the other.”

That relationship is what ultimately creates the GP posterior.

### The RBF kernel

The first kernel you must understand completely is the **Radial Basis Function (RBF)** kernel.

Its simplest one-dimensional form is:

$$
\boxed{
k(x,x')
=
\sigma_f^2
\exp\left(
-\frac{(x-x')^2}{2\ell^2}
\right)
}
$$

There are three important quantities.

### $x$

First input.

### $x'$

Second input.

### $\ell$

The **lengthscale**.

### $\sigma_f^2$

The **output variance / signal variance**.

---

### Understand the lengthscale intuitively

The most important parameter is:

$$
\ell
$$

Think of it as:

> **How far apart can two points be before the GP considers them substantially different?**

Consider:

$$
\ell=0.1
$$

versus

$$
\ell=1.0.
$$

With a small lengthscale:

$$
\ell=0.1
$$

even a relatively small change in $x$ causes covariance to fall rapidly.

The GP becomes:

> very local / rapidly varying.

With a large lengthscale:

$$
\ell=1.0
$$

points remain correlated over a larger region.

The GP becomes:

> smoother / slowly varying.

---

### Numerical example

Take:

$$
\sigma_f^2=1
$$

and

$$
\ell=0.5.
$$

Suppose

$$
x=0.2
$$

and

$$
x'=0.3.
$$

Then:

$$
x-x'=-0.1.
$$

Therefore:

$$
k(0.2,0.3)
=
\exp
\left[
-\frac{(-0.1)^2}{2(0.5)^2}
\right].
$$

Calculate:

$$
0.1^2=0.01
$$

and

$$
2(0.5)^2=0.5.
$$

Therefore:

$$
k=e^{-0.02}\approx0.9802.
$$

So:

$$
\boxed{k(0.2,0.3)\approx0.980}
$$

Very high similarity.

---

### What happens farther away?

Now:

$$
x=0.2
$$

and

$$
x'=0.8.
$$

Distance:

$$
|x-x'|=0.6.
$$

Therefore:

$$
k(0.2,0.8)
=
\exp
\left(
-\frac{0.6^2}{2(0.5)^2}
\right)
$$

$$
=
\exp(-0.72)
$$

$$
\approx0.4868.
$$

So:

$$
\boxed{k(0.2,0.8)\approx0.487}
$$

The points are considerably less correlated.

---

### Very far away

Suppose:

$$
x=0.2,\qquad x'=2.0.
$$

Then:

$$
|x-x'|=1.8.
$$

$$
k
=
\exp
\left(
-\frac{1.8^2}{2(0.5)^2}
\right)
$$

$$
=
\exp(-6.48)
$$

$$
\approx0.00153.
$$

Therefore:

$$
\boxed{k\approx0.0015}
$$

Almost no correlation.

---

### The most important special case: $x=x'$

What happens when the two points are identical?

$$
x=x'.
$$

Then:

$$
x-x'=0.
$$

Therefore:

$$
k(x,x)
=
\sigma_f^2e^0
$$

and

$$
\boxed{k(x,x)=\sigma_f^2}
$$

If:

$$
\sigma_f^2=1,
$$

then:

$$
k(x,x)=1.
$$

This is the maximum covariance for this RBF kernel.

---

### Why the kernel produces a covariance matrix

Suppose we have three experimental points:

$$
X=
\begin{bmatrix}
x_1\\
x_2\\
x_3
\end{bmatrix}.
$$

The kernel evaluates every pair.

We obtain:

$$
K=
\begin{bmatrix}
k(x_1,x_1)&k(x_1,x_2)&k(x_1,x_3)\\
k(x_2,x_1)&k(x_2,x_2)&k(x_2,x_3)\\
k(x_3,x_1)&k(x_3,x_2)&k(x_3,x_3)
\end{bmatrix}.
$$

This is the **kernel matrix**, or **covariance matrix**.

---

### A complete 3-point example

Let

$$
X=
\begin{bmatrix}
0\\
0.5\\
1
\end{bmatrix}
$$

and:

$$
\ell=0.5,\qquad\sigma_f^2=1.
$$

We calculate every pair.

#### Point 1 with itself

$$
k(0,0)=1.
$$

#### Point 1 and point 2

$$
k(0,0.5)
=
e^{-0.5^2/(2(0.5)^2)}
$$

$$
=e^{-0.5}
$$

$$
\approx0.6065.
$$

#### Point 1 and point 3

$$
k(0,1)=e^{-2}
$$

$$
\approx0.1353.
$$

#### Point 2 with itself

$$
k(0.5,0.5)=1.
$$

#### Point 2 and point 3

$$
k(0.5,1)=0.6065.
$$

Therefore:

$$
\boxed{
K=
\begin{bmatrix}
1&0.6065&0.1353\\
0.6065&1&0.6065\\
0.1353&0.6065&1
\end{bmatrix}
}
$$

This matrix is one of the central objects in GP regression.

---

### Why is the matrix symmetric?

Notice:

$$
k(x_1,x_2)=k(x_2,x_1).
$$

Therefore:

$$
K_{ij}=K_{ji}.
$$

So:

$$
K=K^T.
$$

This is called a **symmetric matrix**.

For a valid covariance kernel, the covariance matrix must also be **positive semidefinite**.

This is crucial.

### Why positive semidefinite matters

For any vector $a$,

$$
a^TKa\geq0.
$$

This guarantees that the covariance matrix corresponds to a valid probability distribution.

If your kernel produced an invalid covariance matrix, the GP would not define a valid multivariate Gaussian distribution.

This is why kernels are not arbitrary similarity functions.

They need mathematical properties that make GP inference possible.

---

### The connection to the multivariate Gaussian

Suppose we have three function values:

$$
f=
\begin{bmatrix}
f(x_1)\\
f(x_2)\\
f(x_3)
\end{bmatrix}.
$$

The GP says:

$$
f\sim\mathcal{N}(m,K).
$$

For zero mean:

$$
f\sim\mathcal{N}(0,K).
$$

Therefore our kernel matrix

$$
K=
\begin{bmatrix}
1&0.6065&0.1353\\
0.6065&1&0.6065\\
0.1353&0.6065&1
\end{bmatrix}
$$

is literally the covariance matrix of the joint Gaussian distribution.

This is the fundamental bridge:

$$
\boxed{
\text{kernel}
\rightarrow
\text{covariance matrix}
\rightarrow
\text{multivariate Gaussian}
\rightarrow
\text{GP posterior}
}
$$

---

###  RBF in multiple dimensions

Your actual electrolyte problem will have multiple variables.

For:

$$
x\in\mathbb{R}^d
$$

the isotropic RBF is:

$$
\boxed{
k(x,x')
=
\sigma_f^2
\exp
\left[
-\frac{1}{2\ell^2}
\sum_{j=1}^{d}
(x_j-x'_j)^2
\right]
}
$$

Notice:

$$
\sum_{j=1}^{d}(x_j-x'_j)^2
$$

is simply the squared Euclidean distance:

$$
\|x-x'\|^2.
$$

So we can write:

$$
\boxed{
k(x,x')
=
\sigma_f^2
\exp
\left(
-\frac{\|x-x'\|^2}{2\ell^2}
\right)
}
$$

---

### Example with two electrolyte variables

Suppose:

$$
x_1=[0.2,0.7]
$$

and

$$
x_2=[0.3,0.8].
$$

Then:

$$
x_1-x_2=[-0.1,-0.1].
$$

Squared distance:

$$
\|x_1-x_2\|^2
=
(-0.1)^2+(-0.1)^2
$$

$$
=0.02.
$$

For:

$$
\ell=0.5,\qquad\sigma_f^2=1,
$$

we get:

$$
k(x_1,x_2)
=
e^{-0.02/(2(0.5)^2)}
$$

$$
=e^{-0.04}
$$

$$
\approx0.9608.
$$

So:

$$
\boxed{k(x_1,x_2)\approx0.961}
$$

---

### But there is a major problem with one shared lengthscale

Suppose your two variables are:

1. LiPF$_6$ concentration
2. additive concentration

Perhaps the objective is extremely sensitive to additive concentration but much less sensitive to LiPF$_6$ concentration.

Using one $\ell$ assumes the same characteristic scale in every dimension.

That is often too restrictive.

This leads us to:

### ARD

---

### ARD — Automatic Relevance Determination

With ARD, every dimension gets its own lengthscale:

$$
\ell_1,\ell_2,\ldots,\ell_d.
$$

The kernel becomes:

$$
\boxed{
k(x,x')
=
\sigma_f^2
\exp
\left[
-\frac12
\sum_{j=1}^{d}
\frac{(x_j-x'_j)^2}{\ell_j^2}
\right]
}
$$

Compare this with the isotropic version:

$$
\frac{(x_j-x'_j)^2}{\ell^2}.
$$

Now every dimension has its own denominator.

---

### Why this is powerful

Suppose:

$$
\ell_1=1.0
$$

and

$$
\ell_2=0.1.
$$

Then:

#### Dimension 1

$$
\frac{(x_1-x'_1)^2}{1^2}
$$

changes relatively slowly.

#### Dimension 2

$$
\frac{(x_2-x'_2)^2}{0.1^2}
$$

changes very rapidly.

Therefore the GP says:

> Small changes in dimension 2 matter much more to the function similarity.

This is why ARD is extremely useful for materials optimization.

---

### But be careful: lengthscale ≠ causal importance

This is an important scientific point for your work.

You should **not automatically say**:

> “The smallest GP lengthscale means this material variable is physically the most important.”

Why?

Because the learned lengthscale depends on:

- input scaling,
- data distribution,
- correlations between variables,
- noise,
- kernel assumptions,
- number of observations,
- prior,
- model misspecification.

For example, if two electrolyte variables are highly correlated experimentally, the GP may assign their lengthscales in ways that are difficult to interpret physically.

So ARD is useful for:

> statistical sensitivity / model structure

but it is not automatically:

> causal scientific importance.

---

### Why normalization matters enormously for ARD

Suppose:

$$
x_1=\text{temperature in °C}
$$

and

$$
x_2=\text{salt concentration in mol/L}.
$$

Their numerical scales might be:

$$
T\in[20,80]
$$

and

$$
c\in[0.5,1.5].
$$

If you use raw values, the numerical scale of temperature is approximately 60 times larger.

The GP could then have difficulty treating the dimensions comparably.

Instead, BoTorch expects standardized/normalized input conventions for its GP models. The current documentation explicitly notes that `SingleTaskGP`'s hyperparameter priors work best when covariates are normalized to the unit cube.

So:

$$
T=50^\circ C
$$

might become:

$$
T_{\mathrm{norm}}
=
\frac{50-20}{80-20}
=
0.5.
$$

Now all dimensions live approximately in:

$$
[0,1].
$$

This makes ARD lengthscales much more meaningful numerically.

### Outputscale

Now consider:

$$
\sigma_f^2.
$$

Suppose:

$$
\sigma_f^2=4.
$$

Then:

$$
k(x,x)=4.
$$

The GP's latent function has a larger amplitude.

If:

$$
\sigma_f^2=0.1,
$$

then:

$$
k(x,x)=0.1.
$$

The latent function varies much less.

Conceptually:

$$
\boxed{
\ell=\text{how quickly similarity decays}
}
$$

while

$$
\boxed{
\sigma_f^2=\text{how large the function variation can be}
}
$$

## GPyTorch implementation
Now let's stop doing the mathematics manually and let GPyTorch calculate the kernel.

In [1]:
import torch
import gpytorch

torch.set_default_dtype(torch.double)

X = torch.tensor([
    [0.0],
    [0.5],
    [1.0],
])

kernel = gpytorch.kernels.RBFKernel()

kernel.lengthscale = 0.5

K = kernel(X).to_dense()

print(K)

/Users/benediktusmadika/Desktop/AI learning/botorch_sdl/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


tensor([[1.0000, 0.6065, 0.1353],
        [0.6065, 1.0000, 0.6065],
        [0.1353, 0.6065, 1.0000]], grad_fn=<RBFCovarianceBackward>)


### Why `.to_dense()`?

Modern GPyTorch frequently represents covariance matrices using **lazy linear operators** rather than immediately constructing dense matrices.

This is important for scalability.

Conceptually:

```python
kernel(X)
```

does not necessarily mean:

> "Give me a normal dense PyTorch matrix immediately."

Instead, it can return an efficient representation of the covariance operator.

Calling:

```python
kernel(X).to_dense()
```

asks for the actual matrix.

For learning, this is very useful.

For large BO problems, blindly converting everything to dense matrices can be expensive.

### Adding outputscale

A bare `RBFKernel` handles the correlation structure.

We can wrap it in:
gpytorch.kernels.ScaleKernel

In [2]:
base_kernel = gpytorch.kernels.RBFKernel()

kernel = gpytorch.kernels.ScaleKernel(base_kernel)

kernel.base_kernel.lengthscale = 0.5
kernel.outputscale = 2.0

K = kernel(X).to_dense()

print(K)

tensor([[2.0000, 1.2131, 0.2707],
        [1.2131, 2.0000, 1.2131],
        [0.2707, 1.2131, 2.0000]], grad_fn=<MulBackward0>)


Now:

$$
k(x,x')
=
2\exp
\left[
-\frac{(x-x')^2}{2(0.5)^2}
\right].
$$

Therefore, the diagonal becomes:

$$
k(x,x)=2.
$$

### ARD in GPyTorch

Now let's make a two-dimensional kernel.

In [5]:
X = torch.tensor([
    [0.0, 0.0],
    [0.5, 0.5],
    [1.0, 1.0],
])

kernel = gpytorch.kernels.RBFKernel(
    ard_num_dims=2 # Learn a separate lengthscale for each of the two input dimensions.
)

kernel.lengthscale = torch.tensor([[0.5, 0.2]])

K = kernel(X).to_dense()

print(K)

print(kernel)
print("Lengthscale:", kernel.lengthscale)

tensor([[1.0000e+00, 2.6649e-02, 5.0435e-07],
        [2.6649e-02, 1.0000e+00, 2.6649e-02],
        [5.0435e-07, 2.6649e-02, 1.0000e+00]], grad_fn=<ExpBackward0>)
RBFKernel(
  (raw_lengthscale_constraint): Positive()
)
Lengthscale: tensor([[0.5000, 0.2000]], grad_fn=<SoftplusBackward0>)


The exact shape can include batch dimensions depending on the model.
This is why never assume a parameter tensor is always shape [d] in GPyTorch.
Batching can add dimensions.

## RBF versus Matérn

The second kernel you need to know is the **Matérn kernel**.

Conceptually:

- RBF → very smooth functions
- Matérn → less smooth functions

The RBF kernel assumes extremely smooth behavior.

The Matérn family provides adjustable smoothness.

A common choice is:

$$
\nu=\frac52.
$$

The Matérn-$5/2$ kernel is:

$$
k(r)
=
\sigma_f^2
\left(
1+\frac{\sqrt5r}{\ell}
+\frac{5r^2}{3\ell^2}
\right)
\exp
\left(
-\frac{\sqrt5r}{\ell}
\right)
$$

where

$$
r=|x-x'|.
$$

You don't need to memorize the formula yet.

You **do** need to understand the conceptual distinction:

$$
\boxed{\text{RBF} \rightarrow \text{very smooth}}
$$

$$
\boxed{\text{Matérn} \rightarrow \text{allows rougher functions}}
$$


### Why this matters for real materials

Imagine electrolyte ionic conductivity as a function of composition.

A real material response may not be infinitely smooth.

There may be:

- phase changes,
- solvation changes,
- viscosity transitions,
- interfacial effects,
- concentration effects,
- nonlinear interactions.

Therefore, a Matérn kernel can sometimes be a more realistic prior than an RBF.

This is one reason modern BoTorch provides kernel utilities beyond a simple RBF. For example, the current documentation provides `get_matern_kernel_with_gamma_prior` and utilities for constructing RBF or Matérn kernels with dimension-scaled priors.


### The covariance matrix enters GP prediction

Now we arrive at the most important connection.

Suppose:

$$
X=
\begin{bmatrix}
x_1\\
x_2\\
x_3
\end{bmatrix}
$$

and we observe:

$$
y=
\begin{bmatrix}
y_1\\
y_2\\
y_3
\end{bmatrix}.
$$

The GP calculates:

$$
K(X,X).
$$

Then suppose we want to predict at a new point $x_* $.

We calculate:

$$
K(X,x_*)
=
\begin{bmatrix}
k(x_1,x_*)\\
k(x_2,x_*)\\
k(x_3,x_*)
\end{bmatrix}.
$$

And:

$$
k(x_*,x_*).
$$

These quantities determine the posterior.

---

#### Posterior mean

For a simple zero-mean GP with observation noise variance $\sigma_n^2$:

$$
\boxed{
\mu_*
=
K_{*X}
\left(
K_{XX}+\sigma_n^2 I
\right)^{-1}
y
}
$$

where:

- $K_{XX}$: training covariance matrix
- $K_{*X}$: covariance between test point and training points
- $I$: identity matrix
- $\sigma_n^2$: observation noise variance
- $y$: observed measurements

This equation is enormously important.

It says:

> Prediction is a weighted combination of observed values.

And what determines the weights?



#### Therefore kernel → prediction

Suppose:

$$
k(x_*,x_1)=0.95
$$

but:

$$
k(x_*,x_3)=0.05.
$$

Then $x_1$ is much more relevant to prediction at $x_*$ than $x_3$.

So the kernel effectively determines:

> Which previous experiments matter for predicting this new experiment?

That is directly relevant to your SDL.

---

#### Kernel → uncertainty

The posterior variance is:

$$
\boxed{
\sigma_*^2
=
k(x_*,x_*)
-
K_{*X}
\left(
K_{XX}+\sigma_n^2 I
\right)^{-1}
K_{X*}
}
$$

This is equally important.

The second term represents the amount of uncertainty removed by the existing observations.

Therefore:

#### Near existing experiments

Covariance is high.

Information transfer is strong.

Posterior uncertainty decreases.

#### Far from existing experiments

Covariance is low.

Information transfer is weak.

Posterior uncertainty remains high.

And this is exactly why BO can explore unknown regions.

### A crucial BO insight

Your acquisition function does **not directly understand chemistry**.

For example:

```python
LogExpectedImprovement(model, best_f=best_f)
```

does not inherently know:

> “This formulation contains LiPF$_6$.”

It sees the model's posterior.

The model's posterior comes from:

$$
\boxed{
\text{data}
\rightarrow
\text{kernel}
\rightarrow
\text{covariance}
\rightarrow
\text{posterior}
}
$$

Then:

$$
\boxed{
\text{posterior}
\rightarrow
\text{acquisition function}
\rightarrow
\text{next experiment}
}
$$

This is the heart of BoTorch.

---

### Now connect it to `SingleTaskGP`

When you write:

```python
model = SingleTaskGP(
    train_X,
    train_Y,
)
```

you are not merely creating a mysterious "GP object."

You are effectively constructing something containing:

```text
SingleTaskGP
│
├── mean_module
│
├── covar_module
│   │
│   └── kernel
│       └── lengthscale
│
├── likelihood
│   └── noise
│
└── outcome_transform
```

The current BoTorch API documents `SingleTaskGP` as an exact GP supporting inferred or known noise, with a configurable `covar_module`.

---

### Inspecting the actual kernel inside BoTorch

In [ ]:
import torch

from botorch.models import SingleTaskGP

torch.set_default_dtype(torch.double)

train_X = torch.tensor([
    [0.1, 0.2],
    [0.4, 0.5],
    [0.8, 0.7],
])

train_Y = torch.tensor([
    [1.0],
    [2.0],
    [1.5],
])

model = SingleTaskGP(train_X, train_Y)

print(model.covar_module)
# You should inspect what BoTorch actually constructed rather than assuming.

ScaleKernel(
  (base_kernel): MaternKernel(
    (lengthscale_prior): GammaPrior()
    (raw_lengthscale_constraint): Positive()
  )
  (outputscale_prior): GammaPrior()
  (raw_outputscale_constraint): Positive()
)


/Users/benediktusmadika/Desktop/AI learning/botorch_sdl/lib/python3.9/site-packages/botorch/models/utils/assorted.py:202: InputDataWarning: Input data is not standardized (mean = tensor([1.5000]), std = tensor([0.5000])). Please consider scaling the input to zero mean and unit variance.
  warnings.warn(msg, InputDataWarning)


### Inspect the learned parameters
Before fitting:

In [7]:
print(model.covar_module.base_kernel.lengthscale)
print(model.covar_module.outputscale)

tensor([[0.6931, 0.6931]], grad_fn=<SoftplusBackward0>)
tensor(0.6931, grad_fn=<SoftplusBackward0>)


After:

In [8]:
from botorch.fit import fit_gpytorch_mll
from gpytorch.mlls import ExactMarginalLogLikelihood

mll = ExactMarginalLogLikelihood(
    model.likelihood,
    model,
)

fit_gpytorch_mll(mll)

ExactMarginalLogLikelihood(
  (likelihood): GaussianLikelihood(
    (noise_covar): HomoskedasticNoise(
      (noise_prior): GammaPrior()
      (raw_noise_constraint): GreaterThan(1.000E-04)
    )
  )
  (model): SingleTaskGP(
    (likelihood): GaussianLikelihood(
      (noise_covar): HomoskedasticNoise(
        (noise_prior): GammaPrior()
        (raw_noise_constraint): GreaterThan(1.000E-04)
      )
    )
    (mean_module): ConstantMean()
    (covar_module): ScaleKernel(
      (base_kernel): MaternKernel(
        (lengthscale_prior): GammaPrior()
        (raw_lengthscale_constraint): Positive()
      )
      (outputscale_prior): GammaPrior()
      (raw_outputscale_constraint): Positive()
    )
  )
)

In [9]:
print("Lengthscale:")
print(model.covar_module.base_kernel.lengthscale)

print("Outputscale:")
print(model.covar_module.outputscale)

print("Noise:")
print(model.likelihood.noise)

Lengthscale:
tensor([[0.3315, 0.3198]], grad_fn=<SoftplusBackward0>)
Outputscale:
tensor(0.6843, grad_fn=<SoftplusBackward0>)
Noise:
Parameter containing:
tensor([0.0790], requires_grad=True)


## What exactly is being optimized?

This is a critical distinction.

Your BO optimizer:

```python
optimize_acqf(...)
```

is **not** optimizing the GP lengthscale.

The GP fitting step:

```python
fit_gpytorch_mll(mll)
```

optimizes model hyperparameters.

Conceptually:

$$
\boxed{
\texttt{fit\_gpytorch\_mll}
\rightarrow
\ell,\sigma_f^2,\sigma_n^2,\ldots
}
$$

while:

```python
optimize_acqf(...)
```

does:

$$
\boxed{
\text{optimize acquisition function}
\rightarrow
x_{\text{next}}
}
$$

These are two completely different optimization problems.

## Your future SDL loop

```text
Initial electrolyte experiments
        ↓
      train_X
      train_Y
        ↓
   SingleTaskGP
        ↓
   Fit GP hyperparameters
        ↓
      Kernel
        ↓
    GP posterior
        ↓
 Acquisition function
        ↓
  optimize_acqf
        ↓
 Candidate electrolyte
        ↓
     SDL experiment
        ↓
 measured ionic conductivity
        ↓
      append data
        ↓
      repeat

```

## One very important practical issue: noise

The covariance of the **latent function** is not the same as the covariance of the **observations**.

We have:

$$
y=f(x)+\epsilon
$$

with:

$$
\epsilon\sim\mathcal{N}(0,\sigma_n^2).
$$

Therefore, the training covariance becomes:

$$
\boxed{
K_y=K_f+\sigma_n^2I
}
$$

where:

$$
K_f=K(X,X).
$$

For our example:

$$
K_f=
\begin{bmatrix}
1&0.6065&0.1353\\
0.6065&1&0.6065\\
0.1353&0.6065&1
\end{bmatrix}
$$

and suppose:

$$
\sigma_n^2=0.01.
$$

Then:

$$
K_y=
\begin{bmatrix}
1.01&0.6065&0.1353\\
0.6065&1.01&0.6065\\
0.1353&0.6065&1.01
\end{bmatrix}.
$$

Notice:

**noise is added only to the diagonal.**

Why?

Because independent measurement errors do not correlate two different observations.

## Why SDL experiments make this particularly important

Suppose your SDL measures ionic conductivity:

$$
\sigma_{\mathrm{ion}}.
$$

You repeat exactly the same electrolyte formulation three times:

$$
x,x,x.
$$

You might obtain:

$$
1.82,\quad 1.76,\quad 1.80.
$$

The variation could come from:

- cell-to-cell variation,
- temperature control,
- electrolyte mixing,
- injection,
- sealing,
- impedance fitting,
- measurement electronics.

That variation is **observation noise**.

A GP should not interpret every tiny variation as a real change in the underlying material function.

Therefore, the likelihood noise parameter is scientifically important.

## A complete kernel demonstration

In [10]:
import torch
import gpytorch

torch.set_default_dtype(torch.double)

# --------------------------------------------------
# 1. Three normalized experimental points
# --------------------------------------------------

X = torch.tensor([
    [0.0],
    [0.5],
    [1.0],
])

# --------------------------------------------------
# 2. RBF kernel
# --------------------------------------------------

rbf = gpytorch.kernels.RBFKernel()

rbf.lengthscale = 0.5

K_rbf = rbf(X).to_dense()

print("RBF covariance:")
print(K_rbf)

# --------------------------------------------------
# 3. Scaled RBF kernel
# --------------------------------------------------

scaled_rbf = gpytorch.kernels.ScaleKernel(
    gpytorch.kernels.RBFKernel()
)

scaled_rbf.base_kernel.lengthscale = 0.5
scaled_rbf.outputscale = 2.0

K_scaled = scaled_rbf(X).to_dense()

print("\nScaled RBF covariance:")
print(K_scaled)

# --------------------------------------------------
# 4. ARD RBF
# --------------------------------------------------

X_2d = torch.tensor([
    [0.0, 0.0],
    [0.5, 0.5],
    [1.0, 1.0],
])

ard_rbf = gpytorch.kernels.RBFKernel(
    ard_num_dims=2
)

ard_rbf.lengthscale = torch.tensor(
    [[0.5, 0.2]]
)

K_ard = ard_rbf(X_2d).to_dense()

print("\nARD RBF covariance:")
print(K_ard)

# --------------------------------------------------
# 5. Inspect lengthscales
# --------------------------------------------------

print("\nARD lengthscales:")
print(ard_rbf.lengthscale)

RBF covariance:
tensor([[1.0000, 0.6065, 0.1353],
        [0.6065, 1.0000, 0.6065],
        [0.1353, 0.6065, 1.0000]], grad_fn=<RBFCovarianceBackward>)

Scaled RBF covariance:
tensor([[2.0000, 1.2131, 0.2707],
        [1.2131, 2.0000, 1.2131],
        [0.2707, 1.2131, 2.0000]], grad_fn=<MulBackward0>)

ARD RBF covariance:
tensor([[1.0000e+00, 2.6649e-02, 5.0435e-07],
        [2.6649e-02, 1.0000e+00, 2.6649e-02],
        [5.0435e-07, 2.6649e-02, 1.0000e+00]], grad_fn=<ExpBackward0>)

ARD lengthscales:
tensor([[0.5000, 0.2000]], grad_fn=<SoftplusBackward0>)


## Very important: kernel choice is a modeling assumption

When you choose:

```python
RBFKernel()
```

you are saying approximately:

> I believe the underlying function varies smoothly with the input.

When you choose a Matérn kernel, you are making a different assumption about smoothness.

When you use ARD, you allow each dimension to have a different characteristic scale.

Therefore:

$$
\boxed{
\text{Kernel = prior assumption about function structure}
}
$$

This is a much deeper understanding than simply:

> "RBF is a kernel used in GP."

## What BoTorch adds on top of GPyTorch

This is another important architecture point.

You are using:

### PyTorch

Tensor computation, autograd, GPU.

↓

### GPyTorch

Gaussian-process machinery:

- kernels
- likelihoods
- covariance matrices
- exact GP inference
- Cholesky / linear algebra

↓

### BoTorch

Bayesian optimization:

- models
- posterior API
- acquisition functions
- acquisition optimization
- constraints
- multi-objective BO
- batch BO
- cost-aware BO
- multi-fidelity BO
- experimentation-oriented workflows

The current BoTorch documentation explicitly describes its GP models as wrappers around GPyTorch models.

This layered architecture will become increasingly important as we move toward your SDL implementation.

## The complete mathematical chain

You should now see the following chain clearly:

$$
\boxed{X}
$$

↓

kernel:

$$
\boxed{k(x_i,x_j)}
$$

↓

covariance matrix:

$$
\boxed{K}
$$

↓

Gaussian distribution:

$$
\boxed{f\sim\mathcal{N}(m,K)}
$$

↓

condition on observations:

$$
\boxed{p(f_*|X,y,x_*)}
$$

↓

posterior:

$$
\boxed{\mu(x_*),\sigma^2(x_*)}
$$

↓

acquisition:

$$
\boxed{\alpha(x)}
$$

↓

optimization:

$$
\boxed{x_{\mathrm{next}}}
$$

↓

SDL:

$$
\boxed{y_{\mathrm{next}}}
$$

↓

new GP training data.

And then the loop repeats.

